# OCR BNP — Pipeline V10 — 1 appel GPU par page + Export Excel
> `Kernel → Restart Kernel and Run All Cells`


## 1. Dépendances

Version préparée : V26 stabilisée (VLM only, prompts spécialisés, retry ciblé, validation simple).

## 0. Note de cadrage — InternVL3 pour l'analyse documentaire

### Présentation générale
**InternVL3** est un modèle de type **Vision-Language Model (VLM)** conçu pour comprendre simultanément des images et du texte. Contrairement à un OCR classique, qui extrait principalement du texte brut, InternVL3 est capable d'interpréter la structure d'un document, la relation entre les champs, les tableaux, les formulaires et le contexte global de la page.

### Intérêt pour un pipeline IDP
Dans un projet d'**Intelligent Document Processing (IDP)**, InternVL3 peut être pertinent pour :
- lire des documents multi-pages ;
- analyser des formulaires structurés ;
- extraire des informations depuis des scans imparfaits ;
- produire des sorties structurées, par exemple en **JSON**.

Cette approche est particulièrement utile lorsque l'on traite des documents bancaires, des annexes, des ordres de virement, des bulletins de paie ou d'autres pièces justificatives où la compréhension du contexte est aussi importante que la lecture du texte.

### Points forts d'InternVL3
- **Robustesse visuelle** : meilleur comportement sur certaines pages inclinées, chargées ou partiellement dégradées ;
- **Bonne compréhension documentaire** : capacité à relier plusieurs éléments visuels dans une même page ;
- **Extraction structurée** : possibilité de demander une sortie normalisée et exploitable dans un pipeline automatisé ;
- **Intérêt pour les tableaux et formulaires complexes** : utile lorsque la page contient plusieurs blocs d'information.

### Limites à garder en tête
InternVL3 n'est pas une solution magique. Sa performance dépend fortement :
- de la qualité du scan ;
- du prétraitement image ;
- du prompt utilisé ;
- de la stabilité du pipeline de génération.

En pratique, même avec InternVL3, il reste recommandé d'appliquer :
- une correction d'inclinaison (*deskew*) ;
- un redimensionnement cohérent des images ;
- des règles métier de validation après extraction.

### Positionnement par rapport à Qwen2.5-VL-7B
Dans ce notebook, le pipeline actuel est basé sur **Qwen2.5-VL-7B-Instruct**, qui offre un très bon compromis entre rapidité et qualité.  
**InternVL3-14B** peut être envisagé si l'objectif prioritaire devient :
- une meilleure robustesse sur les pages difficiles ;
- une lecture plus tolérante aux scans imparfaits ;
- une meilleure gestion des documents visuellement complexes.

En revanche, ce changement implique généralement :
- davantage de mémoire GPU ;
- un temps d'inférence plus élevé ;
- quelques adaptations techniques dans le notebook.

### Conclusion
InternVL3 constitue une option sérieuse pour renforcer un pipeline documentaire lorsqu'on rencontre des limites sur des pages complexes ou mal scannées.  
Le bon choix ne dépend toutefois pas uniquement du modèle : la performance finale repose sur l'ensemble du pipeline, depuis le rendu PDF jusqu'à la validation métier des données extraites.

In [ ]:
%pip install -q pymupdf pillow transformers openpyxl
print('✅ OK')

## 2. Imports

In [ ]:
import os, time, json, re, random
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

SEED = 42
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

print('✅ Imports + seeds OK')

## 3. Config

In [ ]:
MODEL_PATH      = "/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main"
DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS  = 500
IMAGE_MAX_SIZE  = 1800
PDF_ZOOM        = 2.5

INPUT_DIR  = Path("/mnt/data/transferts_in")
OUTPUT_DIR = Path("/mnt/data/transferts_out")
DEBUG_DIR  = OUTPUT_DIR / "debug_pages"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
EXCEL_PATH = OUTPUT_DIR / f"audit_transferts_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

pdfs = sorted(INPUT_DIR.glob("*.pdf"))
print(f'Device    : {DEVICE}')
print(f'Dossiers  : {len(pdfs)}')
print(f'Excel     : {EXCEL_PATH}')
print(f'Debug PNG : {DEBUG_DIR}')

## 4. Chargement modèle

In [ ]:
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16,
    trust_remote_code=True, low_cpu_mem_usage=True,
)
model.eval().to(DEVICE)
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')

## 5. Utilitaires PDF

In [ ]:
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    r = max_side / max(w, h)
    new_w = max(1, round(w * r))
    new_h = max(1, round(h * r))
    return img.resize((new_w, new_h), Image.LANCZOS)


def is_blank(image, white_threshold=245, blank_ratio=0.995, min_std=8) -> bool:
    arr = np.array(image.convert('L'))
    white_ratio = (arr >= white_threshold).mean()
    std = arr.std()
    return white_ratio >= blank_ratio and std < min_std


def pdf_to_pages(path: Path, zoom=PDF_ZOOM, save_debug=True) -> list:
    doc    = fitz.open(path)
    matrix = fitz.Matrix(zoom, zoom)
    pages  = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        img = resize(Image.frombytes('RGB', [pix.width, pix.height], pix.samples))
        if save_debug:
            img.save(DEBUG_DIR / f"{path.stem}_page_{i+1}.png")
        pages.append({'index': i, 'image': img})
    doc.close()
    return pages


def ask(prompt: str, image: Image.Image, max_new_tokens=MAX_NEW_TOKENS) -> str:
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text',  'text':  prompt},
    ]}]
    text_in = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(
        text=[text_in], images=[image], return_tensors='pt'
    )
    inputs = {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
        )

    generated = out[0][inputs['input_ids'].shape[1]:]
    return processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )


def parse_json(text: str) -> dict:
    if not text:
        return {}
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return {}

    candidate = text[start:end+1]
    try:
        return json.loads(candidate)
    except Exception:
        pass

    for i in range(len(text)):
        if text[i] == "{":
            for j in range(len(text), i, -1):
                if text[j-1] == "}":
                    try:
                        return json.loads(text[i:j])
                    except Exception:
                        continue
    return {}


def ask_json(prompt: str, image: Image.Image, max_new_tokens=MAX_NEW_TOKENS) -> dict:
    return parse_json(ask(prompt, image, max_new_tokens=max_new_tokens))


print('✅ Utilitaires PDF/VLM OK')

## 6. Normalisation des données

In [ ]:
def norm_str(v) -> str:
    """Nettoie une chaîne : espaces multiples, strip."""
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null', 'none', 'n/a') else None


def norm_upper(v) -> str:
    s = norm_str(v)
    return s.upper() if s else None


def norm_compte(v) -> str:
    """Numéro de compte : supprime tous les espaces et caractères non alphanumériques."""
    s = norm_str(v)
    if not s: return None
    return re.sub(r'[^A-Za-z0-9]', '', s).upper()


def norm_montant(v) -> float:
    """Convertit un montant texte en float.
    Gère : '1 261 600.00' / '1,261,600.00' / '1 261 600,00'
    """
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    s = re.sub(r'[^\d.,]', '', s)        # garder chiffres, point, virgule
    if not s: return None
    # Format français : 1 261 600,00
    if s.count(',') == 1 and '.' not in s:
        s = s.replace(',', '.')
    # Format mixte : 1,261,600.00
    elif '.' in s and ',' in s:
        s = s.replace(',', '')
    # Virgules comme séparateurs milliers : 1,261,600
    elif s.count(',') > 1:
        s = s.replace(',', '')
    try:
        return float(s)
    except:
        return None


def norm_date(v) -> str:
    """Normalise une date au format DD/MM/YYYY si possible."""
    if not v: return None
    s = norm_str(v)
    if not s: return None
    # Déjà au bon format
    if re.match(r'^\d{2}/\d{2}/\d{4}$', s): return s
    # Format avec tirets : 2026-02-24 → 24/02/2026
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2})$', s)
    if m: return f'{m.group(3)}/{m.group(2)}/{m.group(1)}'
    return s


def norm_periode(v) -> str:
    """Normalise la période : JANVIER2026PART1 / JANVIER.2026 → JANVIER 2026 PART 1"""
    if not v: return None
    s = str(v).upper().strip()
    # Remplacer séparateurs par espace
    s = re.sub(r'[._\-]', ' ', s)
    # Coller PART au chiffre : PART1 → PART 1
    s = re.sub(r'PART\s*(\d)', r'PART \1', s)
    # Normaliser espaces
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def normalise_doc(doc_type: str, data: dict) -> dict:
    """Applique les normalisations selon le type de document."""
    d = dict(data)  # copie

    if doc_type == 'OV':
        # ── Aplatissement robuste ────────────────────────────────────────────
        # Le modèle peut retourner les données sous 3 formes :
        #   1. Plat    : {"monnaie": "DZD", ...}
        #   2. Zones   : {"zone_32": {"monnaie": "DZD"}, "zone_50": {...}}
        #   3. Mixte   : {"monnaie": "DZD", "zone_50": {"compte_debiteur": "..."}}
        # On aplatit tout en une passe avant de normaliser.
        for zone_key in list(d.keys()):
            if zone_key.startswith('zone_') and isinstance(d[zone_key], dict):
                d.update(d.pop(zone_key))
            elif isinstance(d.get(zone_key), dict):
                # Sous-dict sans préfixe zone_ → aplatir aussi
                d.update(d.pop(zone_key))

        # Aplatir les sous-dicts de zones si le modèle les a retournés groupés
        # ex: {'zone_32': {'monnaie': 'DZD'}} -> {'monnaie': 'DZD'}
        for zone_key in ['zone_32', 'zone_70', 'zone_50', 'zone_59', 'zone_57']:
            sub = d.pop(zone_key, None)
            if isinstance(sub, dict):
                d.update(sub)

        d['monnaie']             = norm_upper(d.get('monnaie'))
        d['montant_chiffres']    = norm_montant(d.get('montant_chiffres'))
        d['montant_lettres']     = norm_str(d.get('montant_lettres'))
        d['periode']             = norm_periode(d.get('periode'))
        d['tranche']             = norm_str(d.get('tranche'))
        d['date_demande']        = norm_date(d.get('date_demande'))
        d['compte_debiteur']     = norm_compte(d.get('compte_debiteur'))
        d['beneficiaire_nom']    = norm_upper(d.get('beneficiaire_nom'))
        d['beneficiaire_compte'] = norm_compte(d.get('beneficiaire_compte'))
        d['beneficiaire_adresse']= norm_str(d.get('beneficiaire_adresse'))
        d['swift']               = norm_compte(d.get('swift'))  # sans espaces
        d['banque_nom']          = norm_upper(d.get('banque_nom'))

    elif doc_type == 'ANNEXE_II':
        d['mois_transfert']    = norm_periode(d.get('mois_transfert'))
        d['nom_prenom']        = norm_upper(d.get('nom_prenom'))
        d['compte_bancaire']   = norm_compte(d.get('compte_bancaire'))
        d['salaire_mensuel']   = norm_montant(d.get('salaire_mensuel'))
        d['nombre_jours']      = norm_str(d.get('nombre_jours'))
        d['part_transferable'] = norm_montant(d.get('part_transferable'))
        d['pays_destination']  = norm_upper(d.get('pays_destination'))
        d['compte_devise']     = norm_str(d.get('compte_devise'))
        d['numero_domiciliation'] = norm_str(d.get('numero_domiciliation'))
        d['date_domiciliation']   = norm_date(d.get('date_domiciliation'))

    elif doc_type == 'ANNEXE_I':
        d['nom_prenom']      = norm_upper(d.get('nom_prenom'))
        d['date_signature']  = norm_date(d.get('date_signature'))

    elif doc_type == 'BULLETIN':
        d['nom_prenom']       = norm_upper(d.get('nom_prenom'))
        d['matricule']        = norm_str(d.get('matricule'))
        d['mois_bulletin']    = norm_periode(d.get('mois_bulletin'))
        d['salaire_base']     = norm_montant(d.get('salaire_base'))
        d['salaire_brut']     = norm_montant(d.get('salaire_brut'))
        d['retenue_ss']       = norm_montant(d.get('retenue_ss'))
        d['retenue_irg']      = norm_montant(d.get('retenue_irg'))
        d['retenue_mutuelle'] = norm_montant(d.get('retenue_mutuelle'))
        d['net_a_payer']      = norm_montant(d.get('net_a_payer'))

    return d


print('✅ Normalisation OK')

## 7. Prompt universel

In [ ]:
PROMPT_TYPE = """Lis cette page bancaire et retourne uniquement un JSON valide.
Choisis un seul type :

- OV        : titre principal "ORDRE DE VIREMENT A L'ETRANGER"
- ANNEXE_I  : titre principal "Annexe I"
- ANNEXE_II : titre principal "Annexe II"
- BULLETIN  : titre principal "BULLETIN DE PAIE"
- AUTRE     : toute autre page

Règles :
- Base-toi d'abord sur le titre principal visible
- Ne fais aucune déduction
- Ne retourne rien d'autre que ce JSON :
{"type":"OV"} ou {"type":"ANNEXE_I"} ou {"type":"ANNEXE_II"} ou {"type":"BULLETIN"} ou {"type":"AUTRE"}
"""


PROMPT_OV = """Lis cette page de type OV et retourne uniquement un JSON valide.

Champs à extraire :
{
  "type": "OV",
  "monnaie": null,
  "montant_chiffres": null,
  "montant_lettres": null,
  "periode": null,
  "tranche": null,
  "date_demande": null,
  "compte_debiteur": null,
  "beneficiaire_nom": null,
  "beneficiaire_compte": null,
  "beneficiaire_adresse": null,
  "swift": null,
  "banque_nom": null
}

Règles :
- Lis uniquement les informations clairement visibles
- Ignore tout ce qui n'est pas explicitement demandé
- Ne fais aucune déduction
- Si un champ est absent ou illisible : null
- Retourne uniquement le JSON
"""


PROMPT_ANNEXE_I = """Lis cette page de type ANNEXE_I et retourne uniquement un JSON valide.

Champs à extraire :
{
  "type": "ANNEXE_I",
  "nom_prenom": null,
  "date_signature": null
}

Règles :
- Lis uniquement les informations clairement visibles
- Ignore tout ce qui n'est pas explicitement demandé
- Ne fais aucune déduction
- Si un champ est absent ou illisible : null
- Retourne uniquement le JSON
"""


PROMPT_ANNEXE_II = """Lis cette page de type ANNEXE_II et retourne uniquement un JSON valide.

Champs à extraire :
{
  "type": "ANNEXE_II",
  "mois_transfert": null,
  "nom_prenom": null,
  "compte_bancaire": null,
  "salaire_mensuel": null,
  "nombre_jours": null,
  "part_transferable": null,
  "pays_destination": null,
  "compte_devise": null,
  "numero_domiciliation": null,
  "date_domiciliation": null
}

Règles :
- Lis uniquement les informations clairement visibles
- Ignore tout ce qui n'est pas explicitement demandé
- Ne fais aucune déduction
- Si un champ est absent ou illisible : null
- Retourne uniquement le JSON
"""


PROMPT_BULLETIN = """Lis cette page de type BULLETIN et retourne uniquement un JSON valide.

Champs à extraire :
{
  "type": "BULLETIN",
  "nom_prenom": null,
  "matricule": null,
  "mois_bulletin": null,
  "salaire_base": null,
  "salaire_brut": null,
  "retenue_ss": null,
  "retenue_irg": null,
  "retenue_mutuelle": null,
  "net_a_payer": null
}

Règles :
- Lis uniquement les informations clairement visibles
- Ignore tout ce qui n'est pas explicitement demandé
- Ne fais aucune déduction
- Si un champ est absent ou illisible : null
- Retourne uniquement le JSON
"""


PROMPTS_EXTRACTION = {
    "OV": PROMPT_OV,
    "ANNEXE_I": PROMPT_ANNEXE_I,
    "ANNEXE_II": PROMPT_ANNEXE_II,
    "BULLETIN": PROMPT_BULLETIN,
}

CRITICAL_FIELDS = {
    "OV": ["compte_debiteur", "montant_chiffres"],
    "ANNEXE_II": ["numero_domiciliation", "date_domiciliation", "part_transferable"],
    "BULLETIN": ["net_a_payer"],
}


def retry_missing_fields(doc_type: str, data: dict, image: Image.Image) -> dict:
    fields = CRITICAL_FIELDS.get(doc_type, [])
    missing = [f for f in fields if not data.get(f)]
    if not missing:
        return data

    prompt_retry = f"""Relis uniquement cette page de type {doc_type}.
Extrais uniquement ces champs : {', '.join(missing)}.

Retourne uniquement un JSON valide avec :
{{
{chr(10).join([f'  "{f}": null' for f in missing])}
}}

Règles :
- Lis uniquement les informations clairement visibles
- Ne fais aucune déduction
- Si un champ est absent ou illisible : null
- Retourne uniquement le JSON
"""

    retry_data = ask_json(prompt_retry, image, max_new_tokens=200)
    for f in missing:
        if retry_data.get(f) not in (None, "", "null", "None"):
            data[f] = retry_data[f]
    return data


def validate_data(doc_type: str, data: dict) -> list:
    errors = []

    if doc_type == "OV":
        compte = data.get("compte_debiteur")
        if compte and len(str(compte)) < 10:
            errors.append("compte_debiteur trop court")
        montant = data.get("montant_chiffres")
        if montant is not None and not isinstance(montant, (int, float)):
            errors.append("montant_chiffres non numérique")

    elif doc_type == "ANNEXE_II":
        montant = data.get("part_transferable")
        if montant is not None and not isinstance(montant, (int, float)):
            errors.append("part_transferable non numérique")

    elif doc_type == "BULLETIN":
        net = data.get("net_a_payer")
        if net is not None and isinstance(net, (int, float)) and net <= 0:
            errors.append("net_a_payer invalide")

    return errors


print('✅ Prompts spécialisés OK')

## 8. Traitement d'un PDF

In [ ]:
def process_pdf(pdf_path: Path, verbose=True) -> dict:
    """
    Traite toutes les pages d'un PDF.
    Retourne {
      'fichier'          : nom du fichier,
      'date_traitement'  : horodatage,
      'pages_trouvees'   : liste des types détectés,
      'pages_manquantes' : types attendus mais absents,
      'anomalies'        : doublons détectés,
      'OV'               : {...},
      'ANNEXE_I'         : {...},
      'ANNEXE_II'        : {...},
      'BULLETIN'         : {...},
    }
    """
    TYPES_ATTENDUS = {'OV', 'ANNEXE_I', 'ANNEXE_II', 'BULLETIN'}

    pages    = pdf_to_pages(pdf_path)
    results  = {}
    doublons = []

    if verbose:
        print(f'\n📁 {pdf_path.name} — {len(pages)} page(s)')

    for page in pages:
        i   = page['index']
        img = page['image']

        if is_blank(img):
            check_blank = ask("Y a-t-il du texte lisible sur cette page ? Réponds uniquement par oui ou non.", img, max_new_tokens=5)
            if 'oui' not in check_blank.lower():
                if verbose:
                    print(f'  Page {i+1} → ignorée (vide)')
                continue

        t0 = time.time()
        type_data = ask_json(PROMPT_TYPE, img, max_new_tokens=40)
        doc_type = type_data.get('type', 'INCONNU')

        if doc_type in ('AUTRE', 'INCONNU', None, ''):
            type_data = ask_json(PROMPT_TYPE, img, max_new_tokens=40)
            doc_type = type_data.get('type', 'INCONNU')

        if doc_type in ('AUTRE', 'INCONNU', None, ''):
            elapsed = time.time() - t0
            if verbose:
                print(f'  Page {i+1} → {str(doc_type):10s} | {elapsed:.1f}s — ignorée')
            continue

        prompt = PROMPTS_EXTRACTION.get(doc_type)
        if not prompt:
            elapsed = time.time() - t0
            if verbose:
                print(f'  Page {i+1} → {doc_type:10s} | {elapsed:.1f}s — type non géré')
            continue

        raw = ask(prompt, img)
        data = parse_json(raw)
        data['type'] = doc_type

        data = retry_missing_fields(doc_type, data, img)
        data = normalise_doc(doc_type, data)
        validation_errors = validate_data(doc_type, data)
        elapsed = time.time() - t0

        if verbose:
            print(f'  Page {i+1} → {doc_type:10s} | {elapsed:.1f}s')
            for k, v in data.items():
                if k != 'type' and v is not None:
                    print(f'    {k:25s}: {v}')
            if validation_errors:
                print(f'    ⚠️  VALIDATION         : {" | ".join(validation_errors)}')

        if doc_type in results:
            doublons.append(doc_type)
            if verbose:
                print(f'    ⚠️  DOUBLON {doc_type} — ignoré')
            continue

        if validation_errors:
            data['status'] = 'INCERTAIN'
            data['validation_errors'] = ' | '.join(validation_errors)

        results[doc_type] = data

    pages_trouvees   = sorted(results.keys())
    pages_manquantes = sorted(TYPES_ATTENDUS - set(results.keys()))

    return {
        'fichier':          pdf_path.name,
        'date_traitement':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'pages_trouvees':   ', '.join(pages_trouvees),
        'pages_manquantes': ', '.join(pages_manquantes) if pages_manquantes else None,
        'anomalies':        'DOUBLON: ' + ', '.join(doublons) if doublons else None,
        **{t: results.get(t, {}) for t in TYPES_ATTENDUS},
    }


print('✅ process_pdf VLM stabilisé OK')

## 9. Export Excel

In [ ]:
# ── Schéma des colonnes par document ─────────────────────────────────────────
SCHEMA = {
    'META': [
        ('fichier',          'Fichier'),
        ('date_traitement',  'Date traitement'),
        ('pages_trouvees',   'Pages trouvées'),
        ('pages_manquantes', 'Pages manquantes'),
        ('anomalies',        'Anomalies'),
    ],
    'OV': [
        ('monnaie',              'Monnaie'),
        ('montant_chiffres',     'Montant (chiffres)'),
        ('montant_lettres',      'Montant (lettres)'),
        ('periode',              'Période'),
        ('tranche',              'Tranche'),
        ('date_demande',         'Date demande'),
        ('compte_debiteur',      'Compte débiteur'),
        ('beneficiaire_nom',     'Bénéficiaire nom'),
        ('beneficiaire_compte',  'Bénéficiaire compte'),
        ('beneficiaire_adresse', 'Bénéficiaire adresse'),
        ('swift',                'SWIFT'),
        ('banque_nom',           'Banque bénéficiaire'),
        ('status',               'Statut'),
        ('validation_errors',    'Erreurs validation'),
    ],
    'ANNEXE_I': [
        ('nom_prenom',     'Nom Prénom'),
        ('date_signature', 'Date signature'),
        ('status',         'Statut'),
        ('validation_errors','Erreurs validation'),
    ],
    'ANNEXE_II': [
        ('mois_transfert',    'Mois transfert'),
        ('nom_prenom',        'Nom Prénom'),
        ('compte_bancaire',   'Compte bancaire'),
        ('salaire_mensuel',   'Salaire mensuel'),
        ('nombre_jours',      'Nombre de jours'),
        ('part_transferable', 'Part transférable'),
        ('pays_destination',  'Pays destination'),
        ('compte_devise',       'Compte devise'),
        ('numero_domiciliation', 'N° domiciliation'),
        ('date_domiciliation',   'Date domiciliation'),
        ('status',               'Statut'),
        ('validation_errors',    'Erreurs validation'),
    ],
    'BULLETIN': [
        ('nom_prenom',       'Nom Prénom'),
        ('matricule',        'Matricule'),
        ('mois_bulletin',    'Mois bulletin'),
        ('salaire_base',     'Salaire base'),
        ('salaire_brut',     'Salaire brut'),
        ('retenue_ss',       'Retenue SS'),
        ('retenue_irg',      'Retenue IRG'),
        ('retenue_mutuelle', 'Retenue mutuelle'),
        ('net_a_payer',      'Net à payer'),
    ],
}

# Couleurs
COLORS = {
    'META':      {'header': 'FF1F4E79', 'col': 'FFD6E4F0'},
    'OV':        {'header': 'FF833C00', 'col': 'FFFCE4D6'},
    'ANNEXE_I':  {'header': 'FF375623', 'col': 'FFE2EFDA'},
    'ANNEXE_II': {'header': 'FF203864', 'col': 'FFDAE3F3'},
    'BULLETIN':  {'header': 'FF3F3151', 'col': 'FFEDE7F6'},
}


def create_excel(path: Path, rows: list):
    """Crée le fichier Excel avec en-têtes colorés et toutes les données."""
    wb = Workbook()
    ws = wb.active
    ws.title = 'Dossiers'

    # ── Construction de la liste ordonnée des colonnes ────────────────────────
    all_cols = []   # (groupe, field_key, label)
    for groupe, cols in SCHEMA.items():
        for field_key, label in cols:
            all_cols.append((groupe, field_key, label))

    # ── Ligne 1 : groupes fusionnés colorés ───────────────────────────────────
    col_idx   = 1
    group_map = defaultdict(list)
    for groupe, _, _ in all_cols:
        group_map[groupe].append(col_idx)
        col_idx += 1

    for groupe, cols in group_map.items():
        s, e = cols[0], cols[-1]
        if s < e:
            ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s)
        c.value     = groupe
        c.font      = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill      = PatternFill('solid', start_color=COLORS[groupe]['header'])
        c.alignment = Alignment(horizontal='center', vertical='center')
    ws.row_dimensions[1].height = 22

    # ── Ligne 2 : noms des colonnes ───────────────────────────────────────────
    for i, (groupe, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i)
        c.value     = label
        c.font      = Font(bold=True, name='Arial', size=9)
        c.fill      = PatternFill('solid', start_color=COLORS[groupe]['col'])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        c.border    = Border(bottom=Side(style='thin'), right=Side(style='hair'))
        ws.column_dimensions[get_column_letter(i)].width = 20
    ws.row_dimensions[2].height = 35
    ws.freeze_panes = ws.cell(row=3, column=len(SCHEMA['META']) + 1)

    # ── Lignes de données ─────────────────────────────────────────────────────
    for row_num, dossier in enumerate(rows, start=3):
        for col_idx, (groupe, field_key, _) in enumerate(all_cols, start=1):
            if groupe == 'META':
                val = dossier.get(field_key)
            else:
                val = dossier.get(groupe, {}).get(field_key)

            c = ws.cell(row=row_num, column=col_idx)
            c.value = val
            c.font  = Font(name='Arial', size=9)
            c.fill  = PatternFill('solid', start_color=COLORS[groupe]['col'])

            # Format nombre pour les montants
            if isinstance(val, float):
                c.number_format = '0.00'

            # Alerte rouge si anomalie
            if groupe == 'META' and field_key == 'anomalies' and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')
            if groupe == 'META' and field_key == 'pages_manquantes' and val:
                c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')

    wb.save(path)
    print(f'✅ Excel sauvegardé : {path}')
    print(f'   {len(rows)} dossier(s) | {len(all_cols)} colonnes')


print('✅ Export Excel OK')

## 10. Pipeline complet

In [ ]:
SEP = '=' * 72
rows    = []
t_total = time.time()

for pdf_path in pdfs:
    print(SEP)
    try:
        result = process_pdf(pdf_path, verbose=True)
        rows.append(result)
        manq = result.get('pages_manquantes')
        anom = result.get('anomalies')
        print(f'  ✅ Docs trouvés  : {result["pages_trouvees"]}')
        if manq: print(f'  ⚠️  Manquants    : {manq}')
        if anom: print(f'  🔴 Anomalies    : {anom}')
    except Exception as e:
        print(f'  ❌ ERREUR : {e}')

# Export Excel
print(f'\n{SEP}')
create_excel(EXCEL_PATH, rows)
elapsed = time.time() - t_total
print(f'⏱  Temps total : {elapsed:.1f}s ({elapsed/max(1,len(rows)):.1f}s/dossier)')
print(f'📊 {len(rows)} dossiers traités')

## 11. Analyse rapide des résultats

In [ ]:
import pandas as pd

df = pd.read_excel(EXCEL_PATH, header=1)
print(f'📊 {len(df)} dossiers')
print()

# Dossiers avec anomalies
col_anom = 'Anomalies'
if col_anom in df.columns:
    anom = df[df[col_anom].notna()]
    print(f'🔴 Anomalies (doublons) : {len(anom)}')
    if len(anom): print(anom[['Fichier', col_anom]].to_string(index=False))
    print()

# Dossiers incomplets
col_manq = 'Pages manquantes'
if col_manq in df.columns:
    manq = df[df[col_manq].notna()]
    print(f'⚠️  Dossiers incomplets : {len(manq)}')
    if len(manq): print(manq[['Fichier', col_manq]].to_string(index=False))
    print()

display(df.head(5))